In [4]:
import deepxde as dde
import torch
import numpy as np


Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.
E0000 00:00:1766274308.031930      76 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1766274308.093038      76 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [6]:

# 1. Geometry and Hyperparameters
geom = dde.geometry.Rectangle([0, 0], [2, 1]) # 2x1 Cantilever
vol_frac = 0.4  # Target volume fraction
p = 3           # SIMP penalization power
E0 = 1.0        # Base Young's Modulus
nu = 0.3        # Poisson's ratio

# Lame parameters
lmbda = E0 * nu / ((1 + nu) * (1 - 2 * nu))
mu = E0 / (2 * (1 + nu))

# 2. Define the Linear Elasticity PDE with Density Penalization
def pde(x, y):
    """
    y[:, 0:1] = u (displacement x)
    y[:, 1:2] = v (displacement y)
    y[:, 2:3] = rho (density)
    """
    u, v, rho = y[:, 0:1], y[:, 1:2], y[:, 2:3]
    
    # Gradients
    du_x = dde.grad.jacobian(y, x, i=0, j=0)
    du_y = dde.grad.jacobian(y, x, i=0, j=1)
    dv_x = dde.grad.jacobian(y, x, i=1, j=0)
    dv_y = dde.grad.jacobian(y, x, i=1, j=1)
    
    # Effective Stiffness (SIMP)
    E_eff = (rho**p) * E0
    
    # Stress components (simplified 2D Plane Stress)
    # sigma_xx = E_eff/(1-nu^2) * (du_x + nu*dv_y) ... etc.
    # For brevity, we implement the equilibrium: div(sigma) = 0
    
    # Define equilibrium residuals here (Residual_x, Residual_y)
    # This usually requires second derivatives of u, v and first of rho
    res_u = dde.grad.hessian(y, x, component=0, i=0, j=0) + ... 
    res_v = dde.grad.hessian(y, x, component=1, i=0, j=0) + ...
    
    return [res_u, res_v]

# 3. Boundary Conditions
def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], 0)

def boundary_right_top(x, on_boundary):
    return on_boundary and np.isclose(x[0], 2) and np.isclose(x[1], 1)

# Fixed support at the left
bc_u = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left, component=0)
bc_v = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left, component=1)

# Point load at the top-right corner
def load_term(x, y, _):
    # This is often handled via a small Neumann boundary or Point Loss
    return dde.grad.jacobian(y, x, i=1, j=0) # dummy placeholder

# 4. Custom TO Loss (Compliance + Volume)
def compliance_loss(x, y):
    u, v, rho = y[:, 0:1], y[:, 1:2], y[:, 2:3]
    # Internal energy density: 0.5 * sigma : epsilon
    # Minimize the integral of (rho^p * energy)
    return ... 

def volume_constraint(x, y):
    rho = y[:, 2:3]
    return torch.mean(rho) - vol_frac

# 5. Model Setup
data = dde.data.PDE(
    geom, pde, [bc_u, bc_v], 
    num_domain=2000, num_boundary=400
)

# Network: 2 inputs (x, y), 3 outputs (u, v, rho)
net = dde.nn.FNN([2] + [64] * 4 + [3], "tanh", "Glorot normal")

# Apply Sigmoid to the density output to keep it in [0, 1]
def apply_output_transform(x, y):
    u = y[:, 0:1]
    v = y[:, 1:2]
    rho = torch.sigmoid(y[:, 2:3]) 
    return torch.cat([u, v, rho], dim=1)

net.apply_output_transform(apply_output_transform)

model = dde.Model(data, net)
model.compile("adam", lr=1e-3, loss_weights=[1, 1, 10, 10]) # Weighted PDE vs BCs
model.train(iterations=10000)

Compiling model...
'compile' took 0.003526 s

Training model...



TypeError: in user code:

    File "/home/vanytnut/miniconda3/envs/dl/lib/python3.11/site-packages/deepxde/model.py", line 246, in outputs_losses_train  *
        True, inputs, targets, auxiliary_vars, self.data.losses_train
    File "/home/vanytnut/miniconda3/envs/dl/lib/python3.11/site-packages/deepxde/model.py", line 227, in outputs_losses  *
        outputs_ = self.net(inputs, training=training)
    File "/home/vanytnut/miniconda3/envs/dl/lib/python3.11/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler  **
        raise e.with_traceback(filtered_tb) from None
    File "/home/vanytnut/miniconda3/envs/dl/lib/python3.11/site-packages/deepxde/nn/tensorflow/fnn.py", line 64, in call
        y = self._output_transform(inputs, y)
    File "/tmp/ipykernel_76/3979522339.py", line 81, in apply_output_transform
        rho = torch.sigmoid(y[:, 2:3])

    TypeError: Exception encountered when calling FNN.call().
    
    [1msigmoid(): argument 'input' (position 1) 